In [ ]:
import warnings
warnings.filterwarnings('ignore')

import importlib
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from glob import glob

from src.objects.stack import Stack
from src.prediction import Predictor
from src.validation import Validator
import src.preprocessing as preprocessing
importlib.reload(preprocessing)

In [ ]:
# === Step 1: Preprocess ===
BATCH_SIZE = 8
RESOLUTION = (256, 256)  # Must match model training resolution
stack = preprocessing.preprocess(BATCH_SIZE, RESOLUTION)

In [ ]:
# === Step 2: Predict Single Grayscale Image ===
print("\n Single Image Prediction")
img = Image.open('data/images/frame1.jpg').convert('L').resize(RESOLUTION)
input_data = np.array(img)

predictor = Predictor(model_path='models/final_generator.pth')
predicted_rgb = predictor.predict(input_data)

plt.imshow(predicted_rgb[0])
plt.title("Colorized Output")
plt.axis("off")
plt.show()

In [ ]:
# === Step 3: Validate on Batch of RGB Images ===
print("\n Batch Validation with PSNR & SSIM")

val_paths = glob('data/validation/**/*.jpg', recursive=True)[:8]
ground_truth_batch = np.stack([
    np.array(Image.open(p).convert('RGB').resize(RESOLUTION)) for p in val_paths
])

validator = Validator(model_path='models/final_generator.pth')
results = validator.validate(validation_data=ground_truth_batch, ground_truth=ground_truth_batch)


In [ ]:
# Print evaluation metrics
print("\nValidation Metrics:")
for key, val in results['metrics'].items():
    print(f"{key}: {val:.4f}")

In [ ]:
# Show predictions
for i, rgb_img in enumerate(results['predictions']):
    plt.imshow(rgb_img)
    plt.title(f"Prediction {i+1}")
    plt.axis("off")
    plt.show()
